# 📊 Phân tích Xu hướng & Tính Mùa vụ của Doanh thu
 
**Tác giả:** Hà Quốc Khánh  
**Mục tiêu:** Phân tích xu hướng và tính mùa vụ của dữ liệu doanh thu lịch sử (2012-2022) để hỗ trợ chiến lược dự báo.

In [12]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
import pandas as pd


## 1. Đọc và Chuẩn bị Dữ liệu

Sử dụng Polars để đọc và xử lý dữ liệu với tốc độ cao. Tính thêm cột Lợi nhuận (`Profit` = `Revenue` - `COGS`).

In [14]:
# Tải và đọc dữ liệu
df = pl.read_csv('../Data/sales.csv')

# chuyển date sang dữ liệu datetime
df = df.with_columns([
    pl.col("Date").str.to_date("%Y-%m-%d"),
    (pl.col("Revenue") - pl.col("COGS")).alias("Profit")
]).sort("Date") #thêm cột lợi nhuận profit=revenue-cogs rồi sắp xếp chúng theo thứ tự thời gian

print(f"Data Range: {df['Date'].min()} to {df['Date'].max()}")
print(df.head())

Data Range: 2012-07-04 to 2022-12-31
shape: (5, 4)
┌────────────┬───────────┬──────────┬───────────┐
│ Date       ┆ Revenue   ┆ COGS     ┆ Profit    │
│ ---        ┆ ---       ┆ ---      ┆ ---       │
│ date       ┆ f64       ┆ f64      ┆ f64       │
╞════════════╪═══════════╪══════════╪═══════════╡
│ 2012-07-04 ┆ 5.1235e6  ┆ 3.9830e6 ┆ 1.1406e6  │
│ 2012-07-05 ┆ 2.7518e6  ┆ 2.1506e6 ┆ 601193.22 │
│ 2012-07-06 ┆ 3.0540e6  ┆ 2.5176e6 ┆ 536396.58 │
│ 2012-07-07 ┆ 2.6679e6  ┆ 2.1082e6 ┆ 559684.32 │
│ 2012-07-08 ┆ 2360851.9 ┆ 1.8086e6 ┆ 552229.11 │
└────────────┴───────────┴──────────┴───────────┘


## 2. Trực quan hóa Xu hướng Doanh thu (Kèm Đường Trung bình Trượt)

Xem xét doanh thu hàng ngày qua các năm. Thêm đường trung bình trượt 30 ngày (30-day moving average) để làm mượt dữ liệu và dễ nhìn xu hướng hơn.

In [ ]:
df_pd = df.to_pandas()  #chuyển sang pandas vẽ cho dễ
df_pd['Revenue_MA30'] = df_pd['Revenue'].rolling(window=30).mean()  #trung bình trượt 30 ngày

fig = px.line(
    df_pd, 
    x='Date', 
    y=['Revenue', 'Revenue_MA30'],
    title='Doanh thu Hàng ngày và Đường Trung bình Trượt 30 Ngày (2012 - 2022)',
    labels={'value': 'VND', 'variable': 'Chỉ số'},
    color_discrete_map={'Revenue': 'rgba(65, 105, 225, 0.4)', 'Revenue_MA30': 'darkorange'} #chọn màu này cho dễ nhìn
)
fig.update_layout(hovermode="x unified")
fig.show()

## 3. Phân rã Chuỗi Thời gian (Seasonal Decomposition)

Sử dụng chu kỳ 365 ngày để bắt tính mùa vụ hàng năm. Trực quan hóa các thành phần bằng Plotly để có thể tương tác.

In [ ]:
# Chuyển đổi sang định dạng Pandas để tương thích với thư viện statsmodels.
df_pd = df.to_pandas().set_index('Date')

# Thực hiện phân tích chuỗi thời gian.
# Sử dụng mô hình cộng làm mốc tham chiếu
result = seasonal_decompose(df_pd['Revenue'], model='additive', period=365) #tìm quy luật theo năm 365 ngày

# Plot the components with Plotly
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    subplot_titles=('Quan sát (Observed)', 'Xu hướng (Trend)', 'Mùa vụ (Seasonal)', 'Nhiễu (Residual)'))

fig.add_trace(go.Scatter(x=result.observed.index, y=result.observed, name='Observed', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=result.trend.index, y=result.trend, name='Trend', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal, name='Seasonal', line=dict(color='green')), row=3, col=1)
fig.add_trace(go.Scatter(x=result.resid.index, y=result.resid, name='Residual', mode='markers', marker=dict(size=3, color='red')), row=4, col=1)

fig.update_layout(height=900, title_text="Phân rã Chuỗi Thời gian của Doanh thu Hàng ngày", hovermode="x unified")
fig.show()

## 4. Phân tích Tính Mùa vụ theo Tháng

Tổng hợp dữ liệu theo tháng để xem trung bình Doanh thu và Lợi nhuận.

In [15]:
df_monthly = df.with_columns([
    pl.col("Date").dt.month().alias("Month"),
    pl.col("Date").dt.year().alias("Year")
])

monthly_avg = df_monthly.group_by("Month").agg([
    pl.col("Revenue").mean().alias("Avg_Revenue"),
    pl.col("Profit").mean().alias("Avg_Profit")
]).sort("Month")

fig = px.bar(
    monthly_avg.to_pandas(), 
    x='Month', 
    y=['Avg_Revenue', 'Avg_Profit'], 
    barmode='group',
    title='Trung bình Doanh thu & Lợi nhuận theo Tháng',
    labels={'value': 'Số tiền (VND)', 'variable': 'Chỉ số'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=1, dtick=1))
fig.show()

## 5. Summary & Key Findings

**1. Trend:** Doanh thu có xu hướng tăng giảm theo từng chu kỳ, đường Trend (từ Seasonal Decomposition) cho thấy có những giai đoạn tăng trưởng mạnh và sau đó chững lại. Đường Moving Average 30 ngày giúp theo dõi xu hướng ngắn hạn rõ ràng hơn và lọc bớt nhiễu.
- **Xu hướng tăng trưởng:** 
  - Doanh thu tăng liên tục từ 2012 và đạt đỉnh vào 2016 (2.1 tỷ VND). 
  - Từ 2017-2019 chứng kiến sự suy thoái khi doanh thu giảm gần 50% (còn 1.13 tỷ). 
  - Đến 2022, doanh thu bắt đầu phục hồi (1.16 tỷ VND) sau 2 năm chạm đáy.

**2. Seasonality:** Dựa vào biểu đồ phân tích theo tháng, các tháng 7, 8, 9 có vẻ là các tháng cao điểm về doanh thu. Đồ thị Seasonal xác nhận sự tồn tại của chu kỳ lập lại hàng năm (chu kỳ 365 ngày).
- **Tính mùa vụ (Seasonality):** 
  - Khác với E-commerce thông thường, cao điểm bán hàng là **Mùa Hè (tháng 4-6)** với doanh thu trung bình ngày (~6.5 triệu VND) cao hơn hẳn mức chung (4.2 triệu VND). 
  - Ngược lại, Q4 và các dịp **Lễ, Tết, Black Friday, 11/11** lại là mùa thấp điểm. Doanh thu những ngày hội mua sắm này thậm chí thấp hơn trung bình năm.
  - 👉 *Insight: Các sản phẩm bán ra mang tính đặc thù mùa hè (như máy lạnh, đồ bơi, du lịch biển).* 
- **Tác động của COVID-19 (2020-2021):** 
  - Giai đoạn dịch bệnh bồi thêm cú đánh khiến doanh thu rơi xuống đáy lịch sử (~2.85 triệu VND/ngày). 
  - Tuy nhiên, khi mở cửa năm 2022, trung bình ngày bật tăng mạnh (3.2 triệu/ngày), vượt cả mức trước dịch (2019).

**3. Residuals (Nhiễu):** Có sự xuất hiện của nhiều điểm dị thường (outliers) trong phần Residual, thể hiện các đợt đột biến doanh thu (ví dụ các đợt Flash Sale, sự cố) không nằm trong quy luật Trend hay Seasonality. Các điểm này có thể ảnh hưởng đến mô hình dự báo.

**4. Profit vs Revenue:** Đường Profit bám khá sát Revenue nhưng việc phân tích tách biệt giúp chúng ta đảm bảo rằng sự tăng trưởng doanh thu không đi kèm với việc chi phí bị đội lên quá cao.


